<a href="https://colab.research.google.com/github/CalculatedContent/xgbwwdata/blob/main/XGBWW_OpenML_RandomBinary_W1W2W7W8W9_Alpha_Checkpoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/CalculatedContent/xgbwwdata/blob/main/XGBWW_OpenML_RandomBinary_W1W2W7W8W9_Alpha_Checkpoint.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## 1) Mount Google Drive + configure run


In [1]:
from pathlib import Path
from datetime import datetime
import os

from google.colab import drive

drive.mount('/content/drive', force_remount=False)

RUN_NAME_BASE = "openml_random_binary_w1w2w7w8w9_alpha"
RUNS_ROOT = Path('/content/drive/MyDrive/xgbww_runs') / RUN_NAME_BASE
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

CHECKPOINT_EVERY_ROUNDS = 1   # evaluate/save every N rounds
MAX_ROUNDS = 10000            # safety cap
TARGET_ALPHA = 2.0            # stop when all tracked W-matrix alphas <= this value
TEST_SIZE = 0.2
RANDOM_STATE = 42
FORCE_FRESH_START = False     # True = ignore prior checkpoints and start over for selected dataset
RESTART_RUNTIME_AFTER_INSTALL = False
REUSE_LAST_MODEL = False      # True = resume last selected OpenML binary dataset

print('Runs root:', RUNS_ROOT)
print('Started at:', datetime.utcnow().isoformat() + 'Z')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runs root: /content/drive/MyDrive/xgbww_runs/openml_random_binary_w1w2w7w8w9_alpha
Started at: 2026-03-22T00:41:35.672763Z


/tmp/ipykernel_15499/4095825795.py:23: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print('Started at:', datetime.utcnow().isoformat() + 'Z')


## 2) Fresh install from source (with optional packages)


In [5]:
import os, sys, importlib, subprocess

# Optional flag; default is no restart
RESTART_RUNTIME_AFTER_INSTALL = globals().get("RESTART_RUNTIME_AFTER_INSTALL", False)

def pip_install(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print(" ".join(cmd))
    subprocess.check_call(cmd)

# Fresh clone each run
subprocess.run(["rm", "-rf", "/content/repo_xgbwwdata", "/content/repo_xgboost2ww"], check=True)
subprocess.check_call([
    "git", "clone",
    "https://github.com/CalculatedContent/xgbwwdata.git",
    "/content/repo_xgbwwdata"
])
subprocess.check_call([
    "git", "clone",
    "https://github.com/CalculatedContent/xgboost2ww.git",
    "/content/repo_xgboost2ww"
])

# Core libs
core_pkgs = [
    "xgboost",
    "weightwatcher",
    "torch",
    "openml",
    "pmlb",
    "scikit-learn",
    "pandas",
    "numpy",
    "scipy",
]
pip_install("-U", "pip", "setuptools", "wheel")
for pkg in core_pkgs:
    pip_install(pkg)

# Install xgboost2ww from source (required for W9 conversion)
pip_install("/content/repo_xgboost2ww")

# KEEL is optional and disabled by default
os.environ.setdefault("XGBWW_ENABLE_KEEL", "0")
print("XGBWW_ENABLE_KEEL =", os.environ["XGBWW_ENABLE_KEEL"])
if os.environ["XGBWW_ENABLE_KEEL"].strip().lower() in {"1", "true", "yes", "on"}:
    try:
        pip_install("keel-ds")
        print("Installed optional package: keel-ds")
    except Exception as e:
        print("Optional package keel-ds unavailable; continuing without it.")
        print(type(e).__name__, e)
else:
    print("Skipping keel-ds install (set XGBWW_ENABLE_KEEL=1 to enable).")

# Install xgbwwdata from source repo.
# First try a normal editable install from the repo root.
# If that still does not expose the module, fall back to putting src/ on sys.path,
# which is the correct import path for a src-layout package in Colab.
xgbww_repo = "/content/repo_xgbwwdata"
xgbww_src = f"{xgbww_repo}/src"

try:
    pip_install("-e", xgbww_repo, "--no-build-isolation", "--no-deps")
except Exception as e:
    print("Editable install of xgbwwdata failed; will try sys.path fallback.")
    print(type(e).__name__, e)

# Make sure the live kernel can see the package even if editable install metadata is flaky in Colab
if xgbww_src not in sys.path:
    sys.path.insert(0, xgbww_src)

importlib.invalidate_caches()
if "xgbwwdata" in sys.modules:
    del sys.modules["xgbwwdata"]

try:
    import xgbwwdata
    print("xgbwwdata imported from:", xgbwwdata.__file__)
except Exception as e:
    print("xgbwwdata import still failed.")
    print("sys.path[0:5] =", sys.path[:5])
    print("Expected package path:", xgbww_src)
    print(type(e).__name__, e)
    raise

if RESTART_RUNTIME_AFTER_INSTALL:
    print("Restart requested. Re-run notebook after runtime restart to continue.")
    os.kill(os.getpid(), 9)

/usr/bin/python3 -m pip install -q -U pip setuptools wheel
/usr/bin/python3 -m pip install -q xgboost
/usr/bin/python3 -m pip install -q weightwatcher
/usr/bin/python3 -m pip install -q torch
/usr/bin/python3 -m pip install -q openml
/usr/bin/python3 -m pip install -q pmlb
/usr/bin/python3 -m pip install -q scikit-learn
/usr/bin/python3 -m pip install -q pandas
/usr/bin/python3 -m pip install -q numpy
/usr/bin/python3 -m pip install -q scipy
/usr/bin/python3 -m pip install -q /content/repo_xgboost2ww
XGBWW_ENABLE_KEEL = 0
Skipping keel-ds install (set XGBWW_ENABLE_KEEL=1 to enable).
/usr/bin/python3 -m pip install -q -e /content/repo_xgbwwdata --no-build-isolation --no-deps
xgbwwdata imported from: /content/repo_xgbwwdata/src/xgbwwdata/__init__.py


## 3) Imports + helpers


In [6]:
import gc
import time
import json
import warnings
from datetime import datetime
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xgboost as xgb
import torch
import weightwatcher as ww

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

from xgbwwdata import Filters, load_dataset
from xgboost2ww import convert

DRIVE_ROOT = None
STATE_PATH = None
METRICS_PATH = None
MODEL_PATH = None
SPLIT_PATH = None
SUMMARY_PATH = None
MODEL_META_PATH = None


def configure_checkpoint_paths(dataset_uid: str):
    global DRIVE_ROOT, STATE_PATH, METRICS_PATH, MODEL_PATH, SPLIT_PATH, SUMMARY_PATH, MODEL_META_PATH
    dataset_id = dataset_uid.split(':', 1)[1]
    DRIVE_ROOT = RUNS_ROOT / f'openml_{dataset_id}_w1w2w7w8w9_alpha'
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    STATE_PATH = DRIVE_ROOT / f'state_openml_{dataset_id}.json'
    METRICS_PATH = DRIVE_ROOT / f'metrics_openml_{dataset_id}.csv'
    MODEL_PATH = DRIVE_ROOT / f'model_openml_{dataset_id}_latest.json'
    SPLIT_PATH = DRIVE_ROOT / f'data_split_openml_{dataset_id}.npz'
    SUMMARY_PATH = DRIVE_ROOT / f'summary_openml_{dataset_id}.json'
    MODEL_META_PATH = DRIVE_ROOT / f'model_meta_openml_{dataset_id}.json'


def _extract_weight_shape(layer):
    candidates = []

    if hasattr(layer, 'weight'):
        candidates.append(layer.weight)

    if hasattr(layer, 'modules'):
        for sublayer in layer.modules():
            if sublayer is layer:
                continue
            if hasattr(sublayer, 'weight'):
                candidates.append(sublayer.weight)

    for w in candidates:
        if hasattr(w, 'detach'):
            shape = tuple(w.detach().cpu().shape)
        else:
            shape = tuple(getattr(w, 'shape', ()))
        if len(shape) == 2:
            return shape

    return None


def layer_min_matrix_dim(layer):
    shape = _extract_weight_shape(layer)
    if shape is None:
        return 0
    return int(min(shape))


def ww_stats_for_matrix(layer, matrix_name):
    watcher = ww.WeightWatcher(model=layer)
    details = watcher.analyze(randomize=True, detX=True, ERG=True, plot=False)
    if 'alpha' not in details.columns:
        raise RuntimeError(f"WeightWatcher output missing alpha for {matrix_name}: columns={list(details.columns)}")

    row = details.iloc[0]
    return {
        'alpha': float(row['alpha']),
        'D': float(row['D']) if 'D' in details.columns and pd.notna(row.get('D')) else np.nan,
        'num_traps': float(row['num_traps']) if 'num_traps' in details.columns and pd.notna(row.get('num_traps')) else np.nan,
        'ERG_gap': float(row['ERG_gap']) if 'ERG_gap' in details.columns and pd.notna(row.get('ERG_gap')) else np.nan,
    }


def convert_matrix_layer(model, Xtr, ytr, matrix_name, train_params=None, num_boost_round=None):
    return convert(
        model,
        Xtr,
        ytr,
        W=matrix_name,
        return_type='torch',
        train_params=train_params,
        num_boost_round=num_boost_round,
    )


def is_small_matrix_error(err):
    msg = str(err)
    return ('minimum of 2 is required by TruncatedSVD' in msg) or ('Found array with 1 feature(s)' in msg)


def evaluate_model(booster, Xtr, ytr, Xte, yte):
    dtr = xgb.DMatrix(Xtr, label=ytr)
    dte = xgb.DMatrix(Xte, label=yte)

    m_tr = booster.predict(dtr, output_margin=True).astype(np.float32)
    p_tr = 1.0 / (1.0 + np.exp(-m_tr))
    tr_acc = float(accuracy_score(ytr, (p_tr >= 0.5).astype(int)))

    m_te = booster.predict(dte, output_margin=True).astype(np.float32)
    p_te = 1.0 / (1.0 + np.exp(-m_te))
    te_acc = float(accuracy_score(yte, (p_te >= 0.5).astype(int)))
    te_loss = float(log_loss(yte, np.vstack([1 - p_te, p_te]).T, labels=[0, 1]))
    return tr_acc, te_acc, te_loss


def save_checkpoint(state, metrics_df, booster):
    STATE_PATH.write_text(json.dumps(state, indent=2))
    metrics_df.to_csv(METRICS_PATH, index=False)
    booster.save_model(str(MODEL_PATH))


## 4) Pick a random binary OpenML dataset, prepare train/test split, and resume if needed


In [7]:
import random
import openml

filters = Filters(min_rows=100, max_rows=None, max_features=100000)
selection_history_path = RUNS_ROOT / 'selection_history.json'

if selection_history_path.exists():
    selection_history = json.loads(selection_history_path.read_text())
else:
    selection_history = {'selected_dataset_uids': [], 'last_selected_dataset_uid': None, 'events': []}

if REUSE_LAST_MODEL and selection_history.get('last_selected_dataset_uid'):
    selected_dataset_uid = selection_history['last_selected_dataset_uid']
else:
    catalog = openml.datasets.list_datasets(output_format='dataframe')
    did_col = 'did' if 'did' in catalog.columns else 'dataset_id'
    class_col = 'NumberOfClasses' if 'NumberOfClasses' in catalog.columns else None
    status_col = 'status' if 'status' in catalog.columns else None

    if class_col is None:
        raise RuntimeError('OpenML catalog missing NumberOfClasses column; cannot pre-filter binary datasets.')

    binary_catalog = catalog[catalog[class_col] == 2].copy()
    if status_col is not None:
        binary_catalog = binary_catalog[binary_catalog[status_col].astype(str).str.lower().isin(['active', 'deactivated'])]

    available_ids = [int(v) for v in binary_catalog[did_col].dropna().astype(int).tolist()]
    previously_selected = set(selection_history.get('selected_dataset_uids', []))
    untested_ids = [did for did in available_ids if f'openml:{did}' not in previously_selected]

    pool = untested_ids if untested_ids else available_ids
    selected_did = random.choice(pool)
    selected_dataset_uid = f'openml:{selected_did}'

configure_checkpoint_paths(selected_dataset_uid)

X, y, meta = load_dataset(selected_dataset_uid, filters=filters)

y = np.asarray(y)
if len(np.unique(y)) != 2:
    raise ValueError(f'Dataset is not binary classification; dataset_uid={selected_dataset_uid}; classes={np.unique(y)}')

if hasattr(X, 'tocsr'):
    X = X.tocsr().astype(np.float32)
else:
    X = np.asarray(X, dtype=np.float32)

if FORCE_FRESH_START:
    for p in [STATE_PATH, METRICS_PATH, MODEL_PATH, SPLIT_PATH, SUMMARY_PATH, MODEL_META_PATH]:
        if p.exists():
            p.unlink()

if SPLIT_PATH.exists():
    npz = np.load(SPLIT_PATH, allow_pickle=False)
    tr_idx, te_idx = npz['tr_idx'], npz['te_idx']
else:
    all_idx = np.arange(len(y))
    tr_idx, te_idx = train_test_split(
        all_idx, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    np.savez_compressed(SPLIT_PATH, tr_idx=tr_idx, te_idx=te_idx)

Xtr, Xte = X[tr_idx], X[te_idx]
ytr, yte = y[tr_idx], y[te_idx]

if isinstance(meta, dict):
    dataset_name = meta.get('dataset_name', meta.get('name', 'unknown'))
    dataset_uid = meta.get('dataset_uid', meta.get('uid', selected_dataset_uid))
else:
    dataset_name = getattr(meta, 'dataset_name', getattr(meta, 'name', 'unknown'))
    dataset_uid = getattr(meta, 'dataset_uid', getattr(meta, 'uid', selected_dataset_uid))

dataset_id = dataset_uid.split(':', 1)[1]

selection_history['last_selected_dataset_uid'] = dataset_uid
if dataset_uid not in selection_history['selected_dataset_uids']:
    selection_history['selected_dataset_uids'].append(dataset_uid)
selection_history['events'].append({'dataset_uid': dataset_uid, 'selected_at_utc': datetime.utcnow().isoformat() + 'Z'})
selection_history_path.write_text(json.dumps(selection_history, indent=2))

model_meta = {
    'dataset_uid': dataset_uid,
    'dataset_id': dataset_id,
    'dataset_name': dataset_name,
    'selected_at_utc': datetime.utcnow().isoformat() + 'Z',
    'runs_root': str(RUNS_ROOT),
    'checkpoint_dir': str(DRIVE_ROOT),
    'checkpoint_files': {
        'state': STATE_PATH.name,
        'metrics': METRICS_PATH.name,
        'model': MODEL_PATH.name,
        'split': SPLIT_PATH.name,
        'summary': SUMMARY_PATH.name,
    },
}
MODEL_META_PATH.write_text(json.dumps(model_meta, indent=2))

print('dataset_name:', dataset_name)
print('dataset_uid:', dataset_uid)
print('X shape:', X.shape, '| train:', Xtr.shape, '| test:', Xte.shape)
print('Checkpoint folder:', DRIVE_ROOT)
print('Model metadata file:', MODEL_META_PATH)


dataset_name: PhishingWebsites_seed_1_nrows_2000_nclasses_10_ncols_100_stratify_True
dataset_uid: openml:44674
X shape: (2000, 68) | train: (1600, 68) | test: (400, 68)
Checkpoint folder: /content/drive/MyDrive/xgbww_runs/openml_random_binary_w1w2w7w8w9_alpha/openml_44674_w1w2w7w8w9_alpha
Model metadata file: /content/drive/MyDrive/xgbww_runs/openml_random_binary_w1w2w7w8w9_alpha/openml_44674_w1w2w7w8w9_alpha/model_meta_openml_44674.json


## 5) Training loop (checkpoint/resume each round) until alpha(W1), alpha(W2), alpha(W7), alpha(W8), and alpha(W9) <= 2


In [ ]:
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'tree_method': 'hist',
    'learning_rate': 0.05,
    'max_depth': 5,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_lambda': 1.0,
    'seed': RANDOM_STATE,
}

state = {
    'dataset_uid': dataset_uid,
    'target_alpha': TARGET_ALPHA,
    'max_rounds': MAX_ROUNDS,
    'checkpoint_every_rounds': CHECKPOINT_EVERY_ROUNDS,
    'last_completed_round': 0,
    'target_round_w1': None,
    'target_round_w2': None,
    'target_round_w7': None,
    'target_round_w8': None,
    'target_round_w9': None,
    'target_round_all': None,
}

if STATE_PATH.exists() and METRICS_PATH.exists() and MODEL_PATH.exists():
    state = json.loads(STATE_PATH.read_text())
    state.setdefault('target_round_w1', None)
    state.setdefault('target_round_w2', None)
    state.setdefault('target_round_w7', None)
    state.setdefault('target_round_w8', None)
    state.setdefault('target_round_w9', None)
    state.setdefault('target_round_all', state.get('target_round_both'))
    df_metrics = pd.read_csv(METRICS_PATH)
    booster = xgb.Booster()
    booster.load_model(str(MODEL_PATH))
    start_round = int(state.get('last_completed_round', 0))
    print(f"Resuming from round {start_round}")
else:
    df_metrics = pd.DataFrame()
    booster = None
    start_round = 0
    print('Starting from scratch')

dtr = xgb.DMatrix(Xtr, label=ytr)

MIN_WW_EIGENVALUES = 10  # Wait until constructed matrix has enough spectral support

start_time = time.time()
for r in range(start_round + 1, MAX_ROUNDS + 1):
    booster = xgb.train(
        params=params,
        dtrain=dtr,
        num_boost_round=1,
        xgb_model=booster,
        verbose_eval=False,
    )

    if r % CHECKPOINT_EVERY_ROUNDS != 0:
        continue

    tr_acc, te_acc, te_loss = evaluate_model(booster, Xtr, ytr, Xte, yte)

    try:
        layer_w1 = convert_matrix_layer(booster, Xtr, ytr, 'W1', train_params=params, num_boost_round=r)
        layer_w2 = convert_matrix_layer(booster, Xtr, ytr, 'W2', train_params=params, num_boost_round=r)
        layer_w7 = convert_matrix_layer(booster, Xtr, ytr, 'W7', train_params=params, num_boost_round=r)
        layer_w8 = convert_matrix_layer(booster, Xtr, ytr, 'W8', train_params=params, num_boost_round=r)
        layer_w9 = convert_matrix_layer(booster, Xtr, ytr, 'W9', train_params=params, num_boost_round=r)
    except ValueError as err:
        if is_small_matrix_error(err):
            state['last_completed_round'] = r
            save_checkpoint(state, df_metrics, booster)
            print(f"round={r:4d} | skipping WW: matrix too small for SVD ({err})")
            continue
        raise

    min_dim_w1 = layer_min_matrix_dim(layer_w1)
    min_dim_w2 = layer_min_matrix_dim(layer_w2)
    min_dim_w7 = layer_min_matrix_dim(layer_w7)
    min_dim_w8 = layer_min_matrix_dim(layer_w8)
    min_dim_w9 = layer_min_matrix_dim(layer_w9)

    if min(min_dim_w1, min_dim_w2, min_dim_w7, min_dim_w8, min_dim_w9) == 0:
        state['last_completed_round'] = r
        save_checkpoint(state, df_metrics, booster)
        print(
            f"round={r:4d} | skipping WW: no 2D weight matrix found "
            f"(types: W1={type(layer_w1).__name__}, W2={type(layer_w2).__name__}, W7={type(layer_w7).__name__}, W8={type(layer_w8).__name__}, W9={type(layer_w9).__name__})"
        )
        continue

    if min(min_dim_w1, min_dim_w2, min_dim_w7, min_dim_w8, min_dim_w9) < MIN_WW_EIGENVALUES:
        state['last_completed_round'] = r
        save_checkpoint(state, df_metrics, booster)
        print(
            f"round={r:4d} | warming up "
            f"(min_dim W1={min_dim_w1}, W2={min_dim_w2}, W7={min_dim_w7}, W8={min_dim_w8}, W9={min_dim_w9}; need >= {MIN_WW_EIGENVALUES})"
        )
        continue

    ww_w1 = ww_stats_for_matrix(layer_w1, 'W1')
    ww_w2 = ww_stats_for_matrix(layer_w2, 'W2')
    ww_w7 = ww_stats_for_matrix(layer_w7, 'W7')
    ww_w8 = ww_stats_for_matrix(layer_w8, 'W8')
    ww_w9 = ww_stats_for_matrix(layer_w9, 'W9')

    alpha_w1 = ww_w1['alpha']
    alpha_w2 = ww_w2['alpha']
    alpha_w7 = ww_w7['alpha']
    alpha_w8 = ww_w8['alpha']
    alpha_w9 = ww_w9['alpha']

    rec = {
        'round': r,
        'alpha_w1': alpha_w1,
        'alpha_w2': alpha_w2,
        'alpha_w7': alpha_w7,
        'alpha_w8': alpha_w8,
        'alpha_w9': alpha_w9,
        'D_w1': ww_w1['D'],
        'D_w2': ww_w2['D'],
        'D_w7': ww_w7['D'],
        'D_w8': ww_w8['D'],
        'D_w9': ww_w9['D'],
        'num_traps_w1': ww_w1['num_traps'],
        'num_traps_w2': ww_w2['num_traps'],
        'num_traps_w7': ww_w7['num_traps'],
        'num_traps_w8': ww_w8['num_traps'],
        'num_traps_w9': ww_w9['num_traps'],
        'ERG_gap_W1': ww_w1['ERG_gap'],
        'ERG_gap_W2': ww_w2['ERG_gap'],
        'ERG_gap_W7': ww_w7['ERG_gap'],
        'ERG_gap_W8': ww_w8['ERG_gap'],
        'ERG_gap_W9': ww_w9['ERG_gap'],
        'train_acc': tr_acc,
        'test_acc': te_acc,
        'test_logloss': te_loss,
        'elapsed_min': (time.time() - start_time) / 60.0,
    }
    df_metrics = pd.concat([df_metrics, pd.DataFrame([rec])], ignore_index=True)

    if state.get('target_round_w1') is None and np.isfinite(alpha_w1) and alpha_w1 <= TARGET_ALPHA:
        state['target_round_w1'] = r
    if state.get('target_round_w2') is None and np.isfinite(alpha_w2) and alpha_w2 <= TARGET_ALPHA:
        state['target_round_w2'] = r
    if state.get('target_round_w7') is None and np.isfinite(alpha_w7) and alpha_w7 <= TARGET_ALPHA:
        state['target_round_w7'] = r
    if state.get('target_round_w8') is None and np.isfinite(alpha_w8) and alpha_w8 <= TARGET_ALPHA:
        state['target_round_w8'] = r
    if state.get('target_round_w9') is None and np.isfinite(alpha_w9) and alpha_w9 <= TARGET_ALPHA:
        state['target_round_w9'] = r
    if state.get('target_round_all') is None and np.isfinite(alpha_w1) and np.isfinite(alpha_w2) and np.isfinite(alpha_w7) and np.isfinite(alpha_w8) and np.isfinite(alpha_w9):
        if alpha_w1 <= TARGET_ALPHA and alpha_w2 <= TARGET_ALPHA and alpha_w7 <= TARGET_ALPHA and alpha_w8 <= TARGET_ALPHA and alpha_w9 <= TARGET_ALPHA:
            state['target_round_all'] = r

    state['last_completed_round'] = r
    save_checkpoint(state, df_metrics, booster)

    print(
        f"round={r:4d} | alpha(W1)={alpha_w1:.3f} alpha(W2)={alpha_w2:.3f} alpha(W7)={alpha_w7:.3f} alpha(W8)={alpha_w8:.3f} alpha(W9)={alpha_w9:.3f} "
        f"| test_acc={te_acc:.4f} logloss={te_loss:.4f}"
    )

    if state.get('target_round_all') is not None:
        print(f"Reached alpha <= {TARGET_ALPHA} for W1, W2, W7, W8, and W9 at round {r}.")
        break

summary = {
    'dataset_uid': dataset_uid,
    'target_alpha': TARGET_ALPHA,
    'target_round_w1': state.get('target_round_w1'),
    'target_round_w2': state.get('target_round_w2'),
    'target_round_w7': state.get('target_round_w7'),
    'target_round_w8': state.get('target_round_w8'),
    'target_round_w9': state.get('target_round_w9'),
    'target_round_all': state.get('target_round_all'),
    'final_round': int(state.get('last_completed_round', 0)),
    'checkpoint_dir': str(DRIVE_ROOT),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))

print('\nSummary:', json.dumps(summary, indent=2))


Starting from scratch


## 6) Review saved metrics and milestone rounds


In [ ]:
if METRICS_PATH.exists():
    df = pd.read_csv(METRICS_PATH)
    display(df.tail(10))

    print('Rows saved:', len(df))
    if len(df):
        for w in ['w1', 'w2', 'w7', 'w8', 'w9']:
            col = f'alpha_{w}'
            if col in df.columns:
                print(f"Best (min) alpha {w.upper()}:", float(df[col].min()))

        for w in ['w1', 'w2', 'w7', 'w8', 'w9']:
            col = f'alpha_{w}'
            if col in df.columns:
                m = df[df[col] <= TARGET_ALPHA]
                print(f"First round alpha({w.upper()}) <= target:", None if m.empty else int(m['round'].iloc[0]))

        required = [f'alpha_{w}' for w in ['w1', 'w2', 'w7', 'w8', 'w9'] if f'alpha_{w}' in df.columns]
        if required:
            mall = df[np.logical_and.reduce([df[c] <= TARGET_ALPHA for c in required])]
            print('First round ALL tracked matrices <= target:', None if mall.empty else int(mall['round'].iloc[0]))
else:
    print('No metrics checkpoint yet.')
